# Blood Pressure

This goal of this experiment is to see how 3 factors effect the blood pressure of islanders.

In [1]:
from tqdm.notebook import tqdm
from api.API import IslandsAPI
import pandas as pd
import numpy as np
from time import time_ns,sleep
from math import prod


from api.tasks import * #Load all tasks
from random import shuffle, choice  # Method to randomize groups

tqdm.pandas() #Initialize tqdm


api = IslandsAPI() #Load api
participants = api.get_study_participants() #Get all participants who are contacts
shuffle(participants) #Shuffle the participants list

treatments = {
    "Temperature (C)" : {
        "-20" : SIT_TEMP_NEG20,
        "5" : SIT_TEMP_5,
        "40" : SIT_TEMP_40,
    },
    "Hydration" : {
        "Drink" : DRINK_WATER_250,
        "NoDrink" : None
    },
    "Salty Snack" : {
        "Snack" : EAT_FRIED_CHIPS,
        "No Snack" : None
    }
}

num_replications = 4
num_treatments = prod([len(group) for name,group in treatments.items()])
num_subsamples = 2

In [2]:
from itertools import product
from random import choices

factors = list(treatments.keys())

levels = [list(treatments[f].items()) for f in factors]

treatment_list = list(product(*levels))

n = num_replications * num_treatments

chosen_people = choices(participants,k=n)

groups = np.array_split(chosen_people, num_treatments)

In [3]:
pd.DataFrame([[person.get_id()] + list((t[0] for t in treatment_list[i]))
 for i, group in enumerate(groups)
 for person in group], columns=(['person_id'] + list(treatments.keys()))).to_csv('blood_pressure_assignments.csv', index=False)

In [4]:
assignments = pd.read_csv('blood_pressure_assignments.csv') #Read csv in

assignments['person_id'] =  assignments['person_id'].map(api.get_person_manager().get_person) # Map all person ids to actual persons

assignments.head() #Display the first 5 columns

,person_id,Temperature (C),Hydration,Salty Snack
0,Person: Id zl9wm65n63,-20,Drink,Eaten
1,Person: Id nembnq2bwy,-20,Drink,Eaten
2,Person: Id cjrncpq7nb,-20,Drink,Eaten
3,Person: Id d6h69eee8d,-20,Drink,Eaten
4,Person: Id 22glxwcm3n,-20,Drink,NaN


In [5]:
def wait_with_progress(endtime_ms : float, description: str):#Create a progress bar until a time is complete
    start = time_ns() // 1_000_000
    total_duration = endtime_ms - start

    with tqdm(total=total_duration, unit="ms", desc=description) as pbar:
        last_elapsed = 0

        while (now := time_ns() // 1_000_000) < endtime_ms:
            elapsed = now - start
            if elapsed > last_elapsed:
                pbar.update(elapsed - last_elapsed)
                last_elapsed = elapsed
            sleep(0.1)

        if last_elapsed < total_duration:
            pbar.update(total_duration - last_elapsed)

In [7]:
assignments = assignments.sample(frac=1) #Shuffle order of tasks

tqdm.pandas(desc="Starting Blood Pressure Task 1")
blood_pressure_init_1 = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_pressure_init_1['end_time'].max(), 'Waiting for Blood Pressure Results 1')


tqdm.pandas(desc="Starting Blood Pressure Task 2")
blood_pressure_init_2 = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_pressure_init_2['end_time'].max(), 'Waiting for Blood Pressure Results 2')

tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())
assignments['base_blood_pressure_1'] = assignments['person_id'].apply(lambda p: p.get_task_results()[1].result())

assignments['base_blood_pressure_2'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())
assignments.head()

Starting Blood Pressure Task 1:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Pressure Results 1:   0%|          | 0/57870.0 [00:00<?, ?ms/s]

Starting Blood Pressure Task 2:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Pressure Results 2:   0%|          | 0/56997.0 [00:00<?, ?ms/s]

Retrieving Blood Pressure Results:   0%|          | 0/48 [00:00<?, ?it/s]

,person_id,Temperature (C),Hydration,Salty Snack,base_blood_pressure_1,base_blood_pressure_2
35,Person: Id 99tjwh9l52,40,Drink,Eaten,120/84 mmHg,123/78 mmHg
43,Person: Id 6n6trakalg,40,NoDrink,Eaten,134/83 mmHg,134/86 mmHg
40,Person: Id dxg4mdzga2,40,NoDrink,Eaten,142/83 mmHg,136/83 mmHg
9,Person: Id nltvrn8r6v,-20,NoDrink,Eaten,122/80 mmHg,123/79 mmHg
39,Person: Id epd3rjfy5r,40,Drink,NaN,136/89 mmHg,139/85 mmHg


In [8]:
def run_task(row, treatment_name):
    if row.isna()[treatment_name]:
        return None
    task : Task = treatments[treatment_name][str(row[treatment_name])]
    if task is None:
        return None
    return row['person_id'].do_task(task)

tqdm.pandas(desc='Starting Sit In Room Task')
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Temperature (C)'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Sitting in environment")



tqdm.pandas(desc="Starting Hydration")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Hydration'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Hydrating")

tqdm.pandas(desc="Starting Salty Snack")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Salty Snack'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Eating Snacks")


for i in range(0,num_subsamples):
    tqdm.pandas(desc=f'Measuring Blood Pressure {i+1}')
    blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood pressure levels
    wait_with_progress(blood_glucose_results['end_time'].max(), f'Waiting for Blood Glucose Results {i+1}')


tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())

for i in range(0,num_subsamples):
    assignments[f'end_blood_pressure_{i+1}'] = assignments['person_id'].apply(lambda p: p.get_task_results()[i].result())

Starting Sit In Room Task:   0%|          | 0/48 [00:00<?, ?it/s]

Sitting in environment:   0%|          | 0/597556.0 [00:00<?, ?ms/s]

Starting Hydration:   0%|          | 0/48 [00:00<?, ?it/s]

Hydrating:   0%|          | 0/23354.89990234375 [00:00<?, ?ms/s]

Starting Salty Snack:   0%|          | 0/48 [00:00<?, ?it/s]

...

Measuring Blood Pressure 1:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Glucose Results 1:   0%|          | 0/57127.0 [00:00<?, ?ms/s]

Measuring Blood Pressure 2:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Glucose Results 2:   0%|          | 0/57538.0 [00:00<?, ?ms/s]

Retrieving Blood Pressure Results:   0%|          | 0/48 [00:00<?, ?it/s]

Data is then saved as a csv: [blood_pressure_results.csv](blood_pressure_results.csv)

In [11]:
assignments['person_id'] = assignments['person_id'].map(lambda x: x.get_id())

for i in range(0,num_subsamples):
    assignments[f'end_blood_pressure_{i+1}'] =  assignments[f'end_blood_pressure_{i+1}'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer


assignments['base_blood_pressure_1'] =  assignments['base_blood_pressure_1'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer
assignments['base_blood_pressure_2'] =  assignments['base_blood_pressure_2'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer
assignments.to_csv("blood_pressure_results.csv",index=False)